# XTTS-v2 Playground

>NOTE: the notebook has been executed on a machine with the following specs:

In [1]:
!winfetch


 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
                                  
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll
 lllllllllllllll   lllllllllllllll

Fazei@LAPTOP-O9V24FCP
---------------------
OS: Windows 11 Home [64-bit]
Host: LENOVO 82K2
Kernel: 10.0.26200.0
Motherboard: LENOVO LNVNB161216
Uptime: 2 hours 56 minutes
Packages: 26 (choco)
Shell: PowerShell v5.1.26100.7920
Resolution: 1536x864
Terminal: Visual Studio Code
CPU: AMD Ryzen 5 5600H with Radeon Graphics @ 3.294GHz
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
GPU: AMD Radeon(TM) Graphics
Memory: 8,92 GiB / 13,86 GiB (64%)
Disk (C:): 442 GiB / 475 GiB (93%)

  

## Setting up a venv (Virtual Environment)

> this notebook uses Python 3.10.0

Before we start, make sure to have a venv (virtual environment) to run this model

this can be achieved in the following ways:

- Using python which will include pip
    ```
    python -m venv .venv-name
    ```

- Using UV which is a package manager for Python written in Rust
    ```
    uv venv .venv-name
    ```

## Python & Packages

To get XTTS-v2 working, make sure that your virtual environment has installed all the requirements in [requirements.txt](./requirements.txt)

this can be achieved in the following ways:

- Using pip (package manager for Python)
    ```
    pip install -r requirements
    ```

- Using uv (package manager for Python written in Rust)
    ```
    uv pip install -r requirements.txt
    ```

## Code Implementation

> NOTE: using this model is strictly tied for non-commercial use! see https://tts-hub.github.io/cpml/ for details

>IMPORTANT NOTE on 11-04-2026: what this broke, why does it have lots of silence in between text???

In [2]:
from TTS.api import TTS
from pathlib import Path
import numpy as np
import torch
import re
import time
import soundfile as sf
from tqdm.notebook import tqdm

a_dir = "../outputs"
input_file = "../tts.txt"
a_output = "xttsv2-output.wav"
speaker_path = "./target/james/thinghost_2_james_64kb_short.wav"

def cuda_availability():
    return "cuda" if torch.cuda.is_available() else "cpu"

def clean_text(f):
    with open(f, 'r') as file:
        return file.read().strip() 

def tts_generator():
    tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2", progress_bar=False).to(cuda_availability())
    
    full_text = clean_text(input_file)
    chunks = [c.strip() for c in full_text.split('.') if c.strip()]

    all_wavs = []
    
    # 2. Use a context manager for reliable notebook updates
    with tqdm(total=len(chunks), desc="Processing Chunks") as pbar:
        for i, chunk in enumerate(chunks):
            start_chunk = time.time()
            
            output = tts.tts(
                text=chunk + ".",
                speaker_wav=speaker_path,
                language="en",
                temperature=0.65,
                repetition_penalty=1.1
            )
            
            all_wavs.append(output)
            
            end_chunk = time.time()
            pbar.set_postfix({"last_chunk": f"{end_chunk - start_chunk:.2f}s"}, refresh=True)
            pbar.update(1)
        

    print("Merging chunks...")
    if all_wavs:
        final_wav = np.concatenate(all_wavs)
        final_path = Path(a_dir) / a_output
        sf.write(final_path, final_wav, 24000)
        print(f"Success! Saved to: {final_path}")
    else:
        print("No text was processed.")


def main():
    tts_generator()

if __name__ == "__main__":
    main()



Processing Chunks:   0%|          | 0/64 [00:00<?, ?it/s]

Merging chunks...
Success! Saved to: ..\outputs\xttsv2-output.wav


## Notes regarding implementation

- The result of this model wil end up in the [output directory](../outputs/)

- Make sure to have a text file that you can put in the generator! As far as testing has gone it should work with .txt files.

- Make sure to have a ```target``` directory where you can place the audio file of the person or character that you would like to mimic. So far I have only tested it with ```.wav``` files, but ```.mp3``` files should also work

> NOTE: If this ever hits production make sure to turn it into configurable classes

## Findings

- The coqui tts is known for being good at copying voices, but it seems that after 5 seconds it starts to slowly hallucinate and add gibberish to the audio output.

- It also has a built in chunk divider (text split), but it seems that it ain't that accurate, I believe that doing it manually might be better, although this hasn't been tested yet.

- It does however mimic the voice rlly well (outside the times it hallucinates). It was quite identical to the referenced audio.

- It is quite slow, maybe a kokoro wav splitting approach would be the best so it seems like it's really reading instead of first processing it and then reading it in one go.